# 10.2 圖片切成小塊後，如何排成序列？


同一張 16×16 圖，要排成一串可依序處理的材料。先切成邊長 4 的小方塊，英文叫 patch：每列 4 塊，共 16 塊；依列從左到右、再到下一列排列。每塊仍保存 RGB 的 3×4×4＝48 個數值。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-10-patch-order.svg")))

圖中的 0–15 是小塊所在的序列位置，不是圖片類別。文字模型的一個輸入單位稱 token；在這裡，一條圖片特徵的位置來自一整塊圖，而不是塊內每個像素都變成一個字。


In [ ]:
import torch
from tiny_perceptron.multimodal import scene, patchify, unpatchify

image = scene()[None]
patches = patchify(image, 4)
restored = unpatchify(patches, 3, 16, 16, 4)
print("小塊序列", tuple(patches.shape))
print("拼回相同", torch.equal(image, restored))

`scene()[None]` 增加一張圖的批次軸。`patchify` 切塊並攤平每塊，得到 `(1,16,48)`。`unpatchify` 按通道 3、高寬 16、塊邊長 4 放回格子；`torch.equal` 印出 True，說明這次往返的每個數值相同。

往返成功核對的是兩個工具是否相容，還不能單獨證明排序符合約定。紙上四張卡若依列應為 `[a,b,c,d]`，切和拼都改成依欄的 `[a,c,b,d]`，仍能還原原圖。因此還要直接核對第二位置是否來自右上卡 b，真圖則核對已知位置與 RGB 通道。

把塊邊長改成 8 時，切和拼的參數都要一起改：每列 2 塊，共 4 塊，每塊 3×8×8＝192 個值，預期 `(1,4,192)` 且仍能拼回。這個切法沒有丟像素；下一節把每塊壓成短特徵時，才涉及哪些差異被保留。

<details>
<summary>回顧與查證</summary>

可回顧：[10.1的批次與RGB軸](https://birdhackor.github.io/tiny-perceptron-vlm/10.1.html)。

</details>
